# Clipwell — free cloud notebook

Download and compress a video on Google's Colab runtime, then download **only the final MP4** to your computer. This is an interactive notebook, not the Clipwell website.

### How to use
1. Sign in to Google and connect to the **free CPU runtime** (no GPU or paid plan needed).
2. Paste one or more public video links into the `video_urls` field, separated by spaces, and choose a quality preset. They process one at a time.
3. Click the play button next to the cell. If Google warns about running a notebook from GitHub, review the code and confirm to run it.
4. Keep this tab open. When each video is ready, allow your browser to download the MP4. If your browser asks to allow multiple downloads, accept it. For large files, you can alternatively find it in Colab's **Files** sidebar and download it from there.
5. After the download finishes, choose **Runtime → Disconnect and delete runtime** to remove the temporary files.

Colab's free resources are limited and not guaranteed. Large downloads and encoding can be slow, interrupted, or fail due to storage/network/time limits. A source and output must both fit temporarily in the cloud runtime. Restarting or disconnecting deletes the result. Time estimates, when available, are approximate and apply to the current download or compression stage, not the full queue. The file may already be efficiently compressed, so re-encoding can reduce quality or produce a larger file. Only process content you have permission to use. Avoid putting private links in a notebook that you share or save.

This notebook runs within the Colab interface; it does not host a public website. Colab prohibits file-hosting or media-serving web services, torrents, and some other activities. See [Colab's FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
# @title 1. Paste your link and run this cell
video_urls = "" # @param {type:"string"}
quality = "balanced" # @param ["high", "balanced", "small"]

import sys
import subprocess
import tempfile
import threading
import time
from pathlib import Path
from shutil import move

urls = video_urls.split()
if not urls or len(urls) > 20:
    raise ValueError("Paste between 1 and 20 video links into video_urls, separated by spaces.")
if quality not in ("high", "balanced", "small"):
    raise ValueError("Choose a valid quality preset.")

print("Setting up Clipwell on the cloud runtime (this can take a minute)...", flush=True)
repo = Path(tempfile.mkdtemp(prefix="clipwell-notebook-code-")) / "repo"
subprocess.run([
    "git", "clone", "--quiet", "--depth", "1", "--branch", "arena/c1987889-download-compress",
    "https://github.com/aaqwzsxedc272-svg/download_compress.git", str(repo),
], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo))
import app

# The same backend handles supported sites and public file-host links.
from google.colab import files
import uuid

succeeded = 0
for index, video_url in enumerate(urls, 1):
    print(f"\nVideo {index} of {len(urls)}", flush=True)
    try:
        app.validate_url(video_url)
        job_id = "colab-" + uuid.uuid4().hex
        app.JOBS[job_id] = {"id": job_id, "url": video_url, "preset": quality,
                            "stage": "queued", "progress": None}
        worker = threading.Thread(target=app.run_job, args=(job_id,), daemon=True)
        worker.start()
        last = None
        last_message = time.monotonic()
        while worker.is_alive():
            with app.LOCK:
                job = dict(app.JOBS[job_id])
            percent, eta = job.get("progress"), job.get("eta_seconds")
            marker = (job["stage"], int(percent // 10) if isinstance(percent, (int, float)) else None)
            if marker != last or time.monotonic() - last_message > 60:
                progress = f" — {percent}%" if percent is not None else "…"
                remaining = f"; approximately {max(1, int((eta + 59) // 60))} min left in this stage" if isinstance(eta, (int, float)) else ""
                print(job["stage"].capitalize() + progress + remaining, flush=True)
                last, last_message = marker, time.monotonic()
            time.sleep(3)
        worker.join()
        with app.LOCK:
            job = dict(app.JOBS[job_id])
        if job["stage"] != "done":
            raise RuntimeError(job.get("error", "The job stopped before it finished."))

        source_mb = job["source_size"] / (1024 * 1024)
        output_mb = job["output_size"] / (1024 * 1024)
        print(f"Finished! Original: {source_mb:.1f} MB; MP4: {output_mb:.1f} MB.", flush=True)
        if output_mb >= source_mb:
            print("This source was already efficient; the result is not smaller.", flush=True)
        result_path = Path("/content") / f"clipwell-{job_id}.mp4"
        move(job["path"], str(result_path))
        print(f"Result: {result_path}. Starting download; also available in the Files sidebar.", flush=True)
        files.download(str(result_path))
        succeeded += 1
    except Exception as exc:
        print(f"Video {index} failed: {exc}", flush=True)

print(f"\nQueue finished: {succeeded} of {len(urls)} videos ready. Keep this tab open until all browser downloads finish.", flush=True)
